<div style="background:#1a1a2e;color:#eee;padding:24px 32px;border-radius:10px;font-family:monospace">
<div style="font-size:11px;color:#aaa;letter-spacing:2px;text-transform:uppercase">Senior GenAI Engineer Programme · Section 1 · Week 1 · Module 2</div>
<div style="font-size:24px;font-weight:700;margin:8px 0 4px">Data Structures, Comprehensions, Iterators &amp; Generators</div>
<div style="font-size:13px;color:#ccc">Tier T0 — Platform Engineer &nbsp;|&nbsp; Anthrolytics &nbsp;|&nbsp; ~5 hours</div>
<div style="font-size:12px;color:#888;margin-top:8px">Prerequisites: Module 01_01_01 — The Python Mental Model</div>
</div>

## How to use this notebook

**The loop that builds the mental model:**
1. **Read** the explanation. Understand the concept before running anything.
2. **Write your prediction** in the `> My prediction:` cell. Edit it — actually type your answer.
3. **Run the cell.** Compare output with your prediction.
4. **Read the sticky phrase** at the end of each concept. Say it out loud once.

The collapsible hints (`<details>`) are there for when you are genuinely stuck. Expand them only after a real attempt.

The reference card at the end is worth a screenshot. The sticky phrases in it are what you say cold in week 3 when Priya asks why the Lambda stopped OOM'ing.

## The brief

**Slack — Priya Mehta (CTO) → you, Tuesday 9:17 AM**

> "Ok, you found the bugs. Good. Now the next problem: we have a 50 GB JSONL file of raw contract metadata sitting in S3. The extraction pipeline needs to process every record — all 40 million of them — on a single Lambda with 512 MB of memory. Your predecessor tried to load it all into a list. Lambda OOM'd after 2 minutes. I need you to figure out how to iterate over that file without ever loading more than a few MB at a time. This is your task for today. Marcus will review your solution at 4 PM."

---

**What you are building by 4 PM:**
- `ingestion/streaming_reader.py` — a generator-based JSONL reader
- `ingestion/chunker.py` — a sliding-window text chunker for RAG preprocessing

Marcus runs a verify block that is his code review rubric. Score 6/6 and both PRs are approved.

**The question Priya is actually asking:** Why did `list(records)` kill the Lambda — and what is the exact mechanism that makes a generator different?

That answer lives in the Part 1 object model you already built. This module is the application of it.

---
## Part 1 — Data structure selection: performance and semantics

At 40 million contract records, choosing the wrong data structure is not a style issue — it is a correctness issue.

### Performance facts you need cold

```
Structure       Lookup (in)     Insert/Append    Notes
─────────────────────────────────────────────────────────────────────
list            O(n)            O(1) amortised   append fast; insert(0,x) is O(n)
set             O(1) avg        O(1) avg         unordered; hashable elements only
dict            O(1) avg        O(1) avg         insertion-ordered since Python 3.7
tuple           O(1) index      immutable        safe dict key; safe default arg
deque           O(1) both ends  O(1) both ends   O(1) appendleft — unlike list
defaultdict     O(1) avg        O(1) avg         auto-inits missing keys
Counter         O(1) avg        O(1) avg         counts hashable items
namedtuple      O(1) field      immutable        named fields, zero runtime overhead
```

### The one decision that dominates at 40M records

```
Checking 40 million contract IDs for duplicates:

  list:  id in processed_list  →  O(n) per check  →  4 hours total
  set:   id in processed_set   →  O(1) per check  →  ~0.04 seconds total
```

**The JS gotcha:** JavaScript developers often use arrays as the default container, just as Python developers use lists. The difference: Python has `set` as a first-class builtin for membership testing. Not knowing it exists means writing O(n) code that works fine at 1000 records and collapses at 40 million.

### The `collections` module — four structures used constantly in the Anthrolytics codebase

```python
from collections import defaultdict, Counter, deque, namedtuple

# defaultdict: auto-initialises missing keys
# Accumulating contract IDs by firm without checking KeyError:
by_firm = defaultdict(list)
for c in contracts:
    by_firm[c["firm"]].append(c["contract_id"])  # no KeyError ever

# Counter: counts hashable items in one pass
from collections import Counter
term_freq = Counter(["indemnification", "warranty", "indemnification", "governing law"])
# Counter({'indemnification': 2, 'warranty': 1, 'governing law': 1})
top3 = term_freq.most_common(3)

# deque: O(1) on both ends — the sliding window structure
window = deque(maxlen=3)
window.append("contract_NDA.pdf")   # [NDA]
window.append("contract_MSA.pdf")   # [NDA, MSA]
window.append("contract_SOW.pdf")   # [NDA, MSA, SOW]
window.append("contract_1.pdf")     # [MSA, SOW, 1]  <- NDA dropped automatically

# namedtuple: immutable record with named fields
ContractRef = namedtuple("ContractRef", ["contract_id", "firm"])
ref = ContractRef("NDA-001", "Brennan & Carson")
ref.contract_id   # "NDA-001"
```

> ⚠️ **Common mistake (JS/Java engineers):** Using `list.append()` and then `list.pop(0)` as a queue. `pop(0)` is O(n) — every remaining element shifts left. Use `collections.deque` for O(1) pops from both ends. This shows up in Anthrolytics code when building a sliding window for RAG chunking.

> **My prediction — Part 1.A:**
>
> ```python
> import time
> contract_ids_list = list(range(1_000_000))
> contract_ids_set  = set(range(1_000_000))
>
> # Timing membership test for the last element
> start = time.perf_counter()
> _ = 999_999 in contract_ids_list
> list_time = time.perf_counter() - start
>
> start = time.perf_counter()
> _ = 999_999 in contract_ids_set
> set_time = time.perf_counter() - start
> ```
>
> Which is faster and roughly how much?
>
> My prediction: ___________

In [ ]:
import time
import sys

contract_ids_list = list(range(1_000_000))
contract_ids_set  = set(range(1_000_000))

# Average over 50 repetitions for stable timing
N = 50
start = time.perf_counter()
for _ in range(N):
    _ = 999_999 in contract_ids_list
list_ms = (time.perf_counter() - start) / N * 1000

start = time.perf_counter()
for _ in range(N):
    _ = 999_999 in contract_ids_set
set_ms = (time.perf_counter() - start) / N * 1000

print(f"list lookup (O(n)): {list_ms:.3f} ms — scans up to 1M elements")
print(f"set  lookup (O(1)): {set_ms:.4f} ms — hash lookup")
print(f"Ratio: set is ~{list_ms/set_ms:.0f}x faster")
print()
print(f"Memory: list = {sys.getsizeof(contract_ids_list):,} bytes")
print(f"        set  = {sys.getsizeof(contract_ids_set):,} bytes  (hash table overhead)")
print()
print("The tradeoff: set is ~100-300x faster for membership; uses ~4x more memory.")
print("At 40M records, the speed difference is the difference between seconds and hours.")

> 🗂️ **Sticky phrase — Part 1:** "Right structure, right operation. `in` a list is O(n). `in` a set is O(1). At scale, this is the only decision that matters."

---

### Exercise 1 — Implementation: legal term frequency counter

Priya needs a frequency map of legal terms across Brennan & Carson's contract corpus to tune the contract tagger. She has passed you a sample.

**Task:** Using `Counter`, build a word frequency map over the contract texts below and find the 5 most common legal terms.

```python
contract_texts = [
    "indemnification clause governing law warranty termination indemnification",
    "force majeure governing law indemnification warranty indemnification",
    "termination warranty governing law force majeure warranty warranty",
    "indemnification governing law force majeure termination warranty",
]
```

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

`Counter` accepts any iterable. You need a flat list of all words across all texts. A list comprehension over `text.split()` for each text, then chained (or a nested comprehension), gives you that flat list. Then `Counter.most_common(n)` returns the top n as a list of `(word, count)` tuples.

</details>

In [ ]:
from collections import Counter

contract_texts = [
    "indemnification clause governing law warranty termination indemnification",
    "force majeure governing law indemnification warranty indemnification",
    "termination warranty governing law force majeure warranty warranty",
    "indemnification governing law force majeure termination warranty",
]

def build_term_frequency(texts: list) -> Counter:
    """Return a Counter of all words across all contract texts."""
    # TODO: implement
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
try:
    freq = build_term_frequency(contract_texts)
    assert isinstance(freq, Counter), f"Expected Counter, got {type(freq).__name__}"
    print("Requirement 1: PASSED — returns a Counter")

    top5 = freq.most_common(5)
    assert len(top5) == 5, f"most_common(5) should return 5 items, got {len(top5)}"
    print("Requirement 2: PASSED — most_common(5) returns 5 items")

    assert top5[0][0] == "indemnification", (
        f"Most common term should be 'indemnification', got '{top5[0][0]}'"
    )
    assert top5[0][1] == 5, f"'indemnification' appears 5 times, got {top5[0][1]}"
    print("Requirement 3: PASSED — 'indemnification' is top term with count 5")

    print(f"Score: 3/3")
    print(f"Top 5: {top5}")
except NotImplementedError:
    print("Not implemented — write your solution in build_term_frequency()")
except AssertionError as e:
    print(f"FAILED: {e}")
except TypeError:
    print("build_term_frequency() returned None — did you forget the return statement?")

---
## Part 2 — Comprehensions: when they help and when they hurt

Comprehensions are Python's combined `filter + map` in a single readable expression.

```python
# List comprehension — produces a list in memory immediately
pending = [c for c in contracts if c["status"] == "pending"]

# Dict comprehension — produces a dict in memory immediately
id_to_status = {c["contract_id"]: c["status"] for c in contracts}

# Set comprehension — produces a set in memory immediately (deduplicates)
firms = {c["firm"] for c in contracts}

# Generator expression — lazy, parentheses not square brackets
# Produces ONE item at a time — no list in memory
pending_gen = (c for c in contracts if c["status"] == "pending")
```

### When comprehensions hurt

| Situation | Use instead |
|---|---|
| Large iterable, need only the first match | `next(x for x in it if cond)` |
| Large output, will iterate only once | generator expression |
| Logic spans 2+ lines | regular `for` loop |
| Nested 3+ levels deep | regular `for` loop |

**The most common waste:**
```python
# WRONG — evaluates all 40M records to find the first pending one
first_pending = [c for c in contracts if c["status"] == "pending"][0]

# RIGHT — evaluates until the first match, then stops
first_pending = next(c for c in contracts if c["status"] == "pending")
```

**The JS gotcha:** JavaScript chains `.filter(fn).map(fn)` as separate passes. Python comprehensions do both in one expression and read differently: the transform is first (`c["contract_id"]`), then the iteration (`for c in contracts`), then the filter (`if c["status"] == "pending"`). Read right-to-left to understand Python comprehensions.

> ⚠️ **Common mistake:** Writing a list comprehension when feeding a single downstream pass. `sum([x**2 for x in data])` allocates the entire list before summing. `sum(x**2 for x in data)` sums lazily — no list. At 40M records the difference is 10 GB vs flat memory.

> **My prediction — Part 2.A:**
>
> ```python
> import sys
> list_comp = [x * 2 for x in range(10_000_000)]
> gen_expr  = (x * 2 for x in range(10_000_000))
>
> print(sys.getsizeof(list_comp))
> print(sys.getsizeof(gen_expr))
> ```
>
> What does each `getsizeof` call print — roughly? Why is one much smaller?
>
> My prediction: ___________

In [ ]:
import sys

list_comp = [x * 2 for x in range(10_000_000)]
gen_expr  = (x * 2 for x in range(10_000_000))

print(f"List comprehension: {sys.getsizeof(list_comp):,} bytes  (~{sys.getsizeof(list_comp)//1_000_000} MB)")
print(f"Generator expression: {sys.getsizeof(gen_expr)} bytes")
print()
print("The list holds 10M integer pointers in a contiguous array — all allocated at once.")
print("The generator holds only its iteration state — a few dozen bytes.")
print("This is the exact mechanism behind Priya's 50 GB problem.")

> 🗂️ **Sticky phrase — Part 2:** "Comprehensions for clarity. Generator expressions for memory. Nested 3-deep? Use a loop."

---

### Exercise 2 — Implementation: pending contract index

Priya needs a fast lookup table so the pipeline can check any contract's status by ID without scanning the full list.

**Task:** Write a dict comprehension that maps `contract_id → status` for **pending contracts only**.

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

Dict comprehension syntax: `{key_expr: value_expr for item in iterable if condition}`. The filter condition goes at the end. The key and value expressions use the loop variable. You want `contract_id` as the key and `status` as the value, filtered to `status == "pending"` only.

</details>

In [ ]:
raw_contracts = [
    {"contract_id": "NDA-001", "firm": "Brennan & Carson",    "status": "pending"},
    {"contract_id": "MSA-002", "firm": "Google LLC",          "status": "processed"},
    {"contract_id": "SOW-003", "firm": "Brennan & Carson",    "status": "pending"},
    {"contract_id": "NDA-004", "firm": "Morrison & Foerster", "status": "failed"},
    {"contract_id": "MSA-005", "firm": "Anthrolytics",        "status": "pending"},
]

def build_pending_index(contracts: list) -> dict:
    """Return {contract_id: status} for pending contracts only."""
    # TODO: one dict comprehension
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
try:
    index = build_pending_index(raw_contracts)
    assert isinstance(index, dict), f"Expected dict, got {type(index).__name__}"
    print("Requirement 1: PASSED — returns a dict")

    assert len(index) == 3, f"3 pending contracts, got {len(index)}"
    print("Requirement 2: PASSED — exactly 3 pending contracts in index")

    assert "NDA-001" in index, "NDA-001 should be in the index"
    assert "MSA-002" not in index, "MSA-002 is processed — should not be included"
    assert "NDA-004" not in index, "NDA-004 is failed — should not be included"
    print("Requirement 3: PASSED — non-pending contracts correctly excluded")

    assert index["NDA-001"] == "pending", f"Value should be 'pending', got '{index['NDA-001']}'"
    print("Requirement 4: PASSED — values are status strings")

    print("Score: 4/4")
    print(f"Index: {index}")
except NotImplementedError:
    print("Not implemented — write your solution in build_pending_index()")
except AssertionError as e:
    print(f"FAILED: {e}")
except TypeError:
    print("build_pending_index() returned None — did you forget the return statement?")

---
## Consolidation pause 1 — after Parts 1 and 2

Step away and answer these without looking at the notebook.

**Q1.** You need to check whether a contract ID has already been processed. You have 10 million processed IDs. Which data structure do you use, and why?

**Q2.** Name two situations where a list comprehension is the wrong tool.

**Q3.** Write the expression that finds the first contract from Brennan & Carson in a list of 5 million contracts without evaluating all of them.

**Q4.** Recall the Part 1 sticky phrase: "Right structure, right operation. `in` a list is ___. `in` a set is ___.  At scale, this is the only ___ that matters."

---

<details>
<summary>Expected answers</summary>

**A1.** A `set`. `id in processed_set` is O(1). `id in processed_list` is O(n). At 10M records, O(n) per check means tens of seconds per record; O(1) means microseconds.

**A2.** (a) When you need only the first matching item — use `next(...)` instead. (b) When you iterate the result only once and the input is large — use a generator expression to avoid allocating all results in memory.

**A3.** `next(c for c in contracts if c["firm"] == "Brennan & Carson")`

**A4.** "Right structure, right operation. `in` a list is **O(n)**. `in` a set is **O(1)**. At scale, this is the only **decision** that matters."

</details>

---
## Part 3 — The iterator protocol: what `for` actually does

Before generators make sense, the protocol must be precise. This is not a detail — it is the mechanism that makes the entire streaming pipeline possible.

**What `for contract in contracts:` does under the hood:**

```
Step 1: Python calls iter(contracts)
        → returns an iterator object (has a __next__ method)

Step 2: Python calls next(iterator) in a loop
        → each call returns the next value

Step 3: When the iterator raises StopIteration
        → the loop ends silently
```

**Visualised for the Brennan & Carson batch:**

```
contracts = ["contract_NDA.pdf", "contract_MSA.pdf", "contract_SOW.pdf"]

for contract in contracts:
             ↓
 iter(contracts)       ──►  list_iterator  (internal pointer at index 0)
                                   │
 next(list_iterator)  ──►  "contract_NDA.pdf"     (pointer → 1)
 next(list_iterator)  ──►  "contract_MSA.pdf"     (pointer → 2)
 next(list_iterator)  ──►  "contract_SOW.pdf"     (pointer → 3)
 next(list_iterator)  ──►  StopIteration raised   (loop ends)
```

**Any object implementing `__iter__` and `__next__` is iterable.** You can build your own:

```python
class ContractBatch:
    def __init__(self, contracts):
        self._contracts = contracts
        self._index = 0

    def __iter__(self):
        return self           # object IS its own iterator

    def __next__(self):
        if self._index >= len(self._contracts):
            raise StopIteration
        contract = self._contracts[self._index]
        self._index += 1
        return contract
```

**The JS gotcha:** JavaScript's `for...of` uses `Symbol.iterator` — the same protocol, different syntax. Python makes the protocol directly inspectable: call `iter()` and `next()` yourself in the REPL and watch the iterator step forward.

> ⚠️ **Common mistake:** Reusing an exhausted iterator. Once `StopIteration` fires, the iterator is permanently done. Calling `next()` again raises `StopIteration` immediately — it does not reset. To iterate again, call `iter()` on the original collection for a fresh iterator.

> **My prediction — Part 3.A:**
>
> ```python
> contracts = ["contract_NDA.pdf", "contract_MSA.pdf"]
> it = iter(contracts)
>
> print(next(it))   # call 1
> print(next(it))   # call 2
> print(next(it))   # call 3 — what happens?
> ```
>
> What does the third `next(it)` call do?
>
> My prediction: ___________

In [ ]:
contracts = ["contract_NDA.pdf", "contract_MSA.pdf"]
it = iter(contracts)

print(next(it))   # "contract_NDA.pdf"
print(next(it))   # "contract_MSA.pdf"

try:
    print(next(it))   # iterator exhausted
except StopIteration:
    print("StopIteration raised — iterator is exhausted")
    print("The for loop catches this silently. Calling next() yourself surfaces it.")

print()
print("To iterate again, create a fresh iterator:")
it2 = iter(contracts)
print(next(it2))  # starts over from the beginning

> 🗂️ **Sticky phrase — Part 3:** "`for` calls `iter()` then keeps calling `next()` until `StopIteration`. Any object that speaks this protocol is iterable."

---

### Exercise 3 — Implementation: `ContractBatch` iterator

Marcus wants to confirm you understand what `for` actually does before you build the generator-based reader. Implement the iterator protocol directly.

**Task:** Implement `ContractBatch` with `__iter__` and `__next__` that yields contract dicts one at a time.

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

`__iter__` should return `self` — the object is its own iterator. `__next__` should check `self._index` against `len(self._contracts)`: if within bounds, return the item at `self._index` and increment the index; otherwise raise `StopIteration`. The index starts at 0 in `__init__`.

</details>

In [ ]:
class ContractBatch:
    """Iterator over a list of contract dicts — implements the iterator protocol."""

    def __init__(self, contracts: list):
        self._contracts = contracts
        self._index = 0

    def __iter__(self):
        # TODO: return self
        ...

    def __next__(self):
        # TODO: return next contract dict, or raise StopIteration
        ...


# ── Verify block ──────────────────────────────────────────────────────────────
sample_contracts = [
    {"contract_id": "NDA-001", "firm": "Brennan & Carson"},
    {"contract_id": "MSA-002", "firm": "Google LLC"},
    {"contract_id": "SOW-003", "firm": "Brennan & Carson"},
]

try:
    batch = ContractBatch(sample_contracts)
    assert iter(batch) is batch, "__iter__ must return self"
    print("Requirement 1: PASSED — __iter__ returns self")

    collected = [c["contract_id"] for c in ContractBatch(sample_contracts)]
    assert collected == ["NDA-001", "MSA-002", "SOW-003"], f"Wrong order: {collected}"
    print("Requirement 2: PASSED — iterates in correct order")

    empty_batch = ContractBatch([])
    try:
        next(empty_batch)
        assert False, "Should have raised StopIteration on empty batch"
    except StopIteration:
        pass
    print("Requirement 3: PASSED — StopIteration raised on empty batch")

    # Exhaustion check
    it = ContractBatch(sample_contracts)
    for _ in it:
        pass
    try:
        next(it)
        assert False, "Should raise StopIteration after exhaustion"
    except StopIteration:
        pass
    print("Requirement 4: PASSED — exhausted iterator raises StopIteration on next()")

    print("Score: 4/4")
except NotImplementedError:
    print("Not implemented — fill in __iter__ and __next__")
except AssertionError as e:
    print(f"FAILED: {e}")
except TypeError as e:
    print(f"TypeError — check __iter__ returns self and __next__ raises StopIteration: {e}")

---
## Part 4 — Generators: lazy evaluation and the 50 GB answer

This is the direct answer to Priya's question. A generator holds one object at a time. That is the mechanism behind flat memory.

### Why generators are memory-efficient — from the Module 1.1 object model

```
List approach (what Priya's predecessor did):
─────────────────────────────────────────────
contracts = list(read_all_records(path))
                 ↓
 Python creates 40 MILLION dict objects simultaneously.
 Each dict: ~250 bytes minimum.
 Total: 40M × 250 = 10 GB.
 Lambda memory limit: 512 MB. OOM at 2 minutes.

Generator approach (what you are building):
─────────────────────────────────────────────
for record in read_jsonl(path):
    process(record)
              ↓
 Python creates ONE dict at a time.
 After process(record) returns, no label points to the old dict.
 Garbage collector reclaims it before the next record is created.
 Memory at any moment: ~1 record (~250 bytes).
 Lambda memory pressure: negligible.
```

### How `yield` works

```python
def read_jsonl(path: str):
    with open(path) as f:
        for line in f:                  # ← file iterator, also lazy
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)      # ← PAUSES here, returns the dict
                                        #   next() call RESUMES here
```

**What `gen = read_jsonl("contracts.jsonl")` does:** Returns a generator object immediately. The function body has NOT run. The file has NOT been opened yet. Nothing happens until `next(gen)` is called.

**Each `next()` call:** Runs the function body until the next `yield`, pauses, returns the yielded value. The function's local state — the file handle, the current position — is preserved between calls.

**Generator expressions:** `(expr for x in iterable if cond)` — the lazy version of a list comprehension. Same semantics as a generator function but written inline.

> ⚠️ **Common mistake:** Calling a generator function and not iterating the result.
> ```python
> gen = read_jsonl("contracts.jsonl")  # does nothing — generator object created
> # You must iterate: for r in gen: ...  or  next(gen)  or  list(gen)
> ```
> The function body runs only when you pull from it. `gen = read_jsonl(...)` creates the generator; it does not execute the body.

> **My prediction — Part 4.A:**
>
> ```python
> import sys
>
> def count_up(n):
>     for i in range(n):
>         yield i
>
> gen = count_up(10_000_000)
>
> print(sys.getsizeof(gen))
> print(type(gen))
> ```
>
> What is `sys.getsizeof(gen)`? What is `type(gen)`?
> How many integers are in memory at this point?
>
> My prediction: ___________

In [ ]:
import sys

def count_up(n):
    for i in range(n):
        yield i

gen = count_up(10_000_000)

print(f"Size of generator object: {sys.getsizeof(gen)} bytes")
print(f"Type: {type(gen)}")
print()
print("How many integers are in memory right now?")
print("ZERO. The function body has not run at all.")
print("count_up(10_000_000) returned a generator object — nothing else.")
print("The first integer is only created when next(gen) is called.")
print()
first = next(gen)
print(f"After first next(gen): {first}")
print("Now exactly 1 integer exists. The generator is paused after yield 0.")

> 🗂️ **Sticky phrase — Part 4:** "A generator holds one object at a time. `yield` pauses. `next()` resumes. Memory stays flat."

---

### Exercise 4 — Implementation: `read_jsonl`

This is the core of the streaming reader you are delivering to Marcus at 4 PM.

**Task:** Write `read_jsonl(path: str)` — a generator that:
- Opens a file and yields one parsed JSON dict per line
- Skips empty lines silently
- On `json.JSONDecodeError`: prints a warning to stderr and continues — does not raise

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

The function must contain `yield` — that is what makes it a generator. `open(path)` returns a lazy file iterator: iterating it gives one line at a time. Strip each line before testing if it is empty (`if not line.strip(): continue`). Wrap `json.loads(line)` in try/except for `json.JSONDecodeError`. Print the warning with `print(f"Warning: ...", file=sys.stderr)` and `continue` to the next line.

</details>

In [ ]:
import json
import sys
import io
import inspect

def read_jsonl(path: str):
    """
    Generator: yields one parsed dict per line from a JSONL file.
    Skips empty lines silently.
    Warns on JSONDecodeError and continues — does not raise.
    """
    # TODO: implement this generator
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
# Uses a helper that applies read_jsonl logic to StringIO (no file needed)
def _apply_read_jsonl_logic(text: str):
    """Test helper: apply read_jsonl parsing logic to a StringIO stream."""
    for line in io.StringIO(text):
        line = line.strip()
        if not line:
            continue
        try:
            yield json.loads(line)
        except json.JSONDecodeError as e:
            print(f"Warning: malformed line: {e}", file=sys.stderr)

test_jsonl = (
    '{"contract_id": "NDA-001", "firm": "Brennan & Carson", "status": "pending"}\n'
    '\n'
    'NOT VALID JSON\n'
    '{"contract_id": "MSA-002", "firm": "Google LLC", "status": "processed"}\n'
)

try:
    assert inspect.isgeneratorfunction(read_jsonl), "read_jsonl must use yield (be a generator function)"
    print("Requirement 1: PASSED — read_jsonl is a generator function")

    records = list(_apply_read_jsonl_logic(test_jsonl))
    assert len(records) == 2, f"Expected 2 valid records, got {len(records)}"
    print("Requirement 2: PASSED — 2 valid records returned (empty + malformed lines skipped)")

    assert records[0]["contract_id"] == "NDA-001"
    assert records[1]["contract_id"] == "MSA-002"
    print("Requirement 3: PASSED — records in correct order")

    print("Score: 3/3")
    print("(Warning about malformed JSON should have printed to stderr above)")
except NotImplementedError:
    print("Not implemented — write your generator in read_jsonl()")
except AssertionError as e:
    print(f"FAILED: {e}")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

---
## Consolidation pause 2 — after Parts 3 and 4

Step away and answer these without looking at the notebook.

**Q1.** What two methods must an object implement to work with `for x in obj:`?

**Q2.** You call `gen = read_jsonl("contracts.jsonl")`. Has the file been opened? Has any line been read?

**Q3.** An iterator is exhausted. You call `next()` on it again. What happens?

**Q4.** Recall the Part 4 sticky phrase: "A generator holds ___ object at a time. `yield` ___. `next()` ___. Memory stays ___."

**Q5.** Priya's predecessor wrote `contracts = list(read_all_records(path))`. Using the object model from Module 1.1: explain in one sentence why this OOM'd the 512 MB Lambda.

---

<details>
<summary>Expected answers</summary>

**A1.** `__iter__` (returns the iterator object) and `__next__` (returns the next value or raises `StopIteration`).

**A2.** No. The function body has not run at all. `gen = read_jsonl(...)` creates the generator object and immediately returns. The `open()` call inside the function has not executed.

**A3.** `StopIteration` is raised immediately, every time. An exhausted iterator does not reset.

**A4.** "A generator holds **one** object at a time. `yield` **pauses**. `next()` **resumes**. Memory stays **flat**."

**A5.** `list(read_all_records(path))` creates a list label pointing to 40M dict objects that all exist simultaneously in memory — each dict ~250 bytes, total ~10 GB — exceeding the Lambda's 512 MB limit.

</details>

---
## Part 5 — Generator composition and `itertools`

Real pipelines are composed. The Anthrolytics ingestion pipeline:

```
read_jsonl(path)                    ← source: one record at a time from S3
    ↓ generator
filter_pending(records)             ← stage: drop non-pending contracts
    ↓ generator
extract_text(records)               ← stage: pull contract body text
    ↓ generator
chunk_text(texts, 512, 64)          ← stage: overlapping chunks for RAG
    ↓ generator
embed_chunks(chunks)                ← sink: POST to embedding API
```

Every stage is a generator. Memory stays flat through the entire pipeline — Priya's Lambda runs on 512 MB regardless of whether the source file is 50 GB or 500 GB.

### The five `itertools` you need

```python
from itertools import islice, chain, groupby, product
from collections import deque

# islice: take the first n items without loading all
first_100 = islice(read_jsonl(path), 100)   # lazy — does not load 40M records
# NEVER do: list(read_jsonl(path))[:100]    # loads everything just to slice

# chain: flatten multiple iterables into one continuous stream
all_records = chain(read_jsonl("batch_1.jsonl"), read_jsonl("batch_2.jsonl"))
# No intermediate list — both files processed as one stream

# groupby: group consecutive items — INPUT MUST BE SORTED BY KEY FIRST
sorted_contracts = sorted(read_jsonl(path), key=lambda c: c["firm"])
for firm, firm_contracts in groupby(sorted_contracts, key=lambda c: c["firm"]):
    count = sum(1 for _ in firm_contracts)   # count without materialising
    print(f"{firm}: {count} contracts")

# product: cartesian product — parameter grid searches
models = ["claude-3-haiku", "claude-3-sonnet"]
temperatures = [0.0, 0.5, 1.0]
for model, temp in product(models, temperatures):  # 6 combinations
    pass
```

### The sliding window — the `chunk_text` building block

```python
from collections import deque

def sliding_window(iterable, size: int):
    """Yields tuples of `size` consecutive items, stride 1."""
    window = deque(maxlen=size)
    for item in iterable:
        window.append(item)          # O(1) — oldest item auto-dropped at maxlen
        if len(window) == size:
            yield tuple(window)
```

`deque(maxlen=size)` automatically drops the oldest item when full — O(1) for both operations. Using `list` with `list[1:]` is O(n) per step — catastrophic at 40M records.

> ⚠️ **Common mistake:** Calling `list()` on a generator mid-pipeline.
> ```python
> # WRONG — materialises all records, defeats the entire pipeline
> chunks = list(chunk_text(list(read_jsonl(path)), 512, 64))
>
> # RIGHT — lazy all the way to the sink
> for chunk in chunk_text(read_jsonl(path), 512, 64):
>     embed(chunk)
> ```

> **My prediction — Part 5.A:**
>
> ```python
> from itertools import islice, count
> result = list(islice(count(0), 5))
> print(result)
> ```
>
> What does `result` contain?
> What would happen if you wrote `list(count(0))` instead?
>
> My prediction: ___________

In [ ]:
from itertools import islice, count

result = list(islice(count(0), 5))
print(f"islice(count(0), 5) = {result}")
print()
print("count(0) is infinite — it yields 0, 1, 2, 3, ... and never raises StopIteration.")
print("islice(gen, 5) wraps it: raises StopIteration after exactly 5 items.")
print()
print("list(count(0)) would run until your machine runs out of RAM.")
print("This is why islice exists: to safely bound an unbounded stream.")
print("Same applies to read_jsonl on a 50 GB file: islice for sampling, never [:n].")

> 🗂️ **Sticky phrase — Part 5:** "Compose generators. Each stage yields one item. `itertools` is the glue."

---

### Exercise 5 — Implementation: `chunk_text` for RAG

The second deliverable for Marcus's 4 PM review: the sliding-window chunker that feeds the embedding pipeline.

**Task:** Write `chunk_text(text: str, chunk_size: int, overlap: int) -> Generator[str, None, None]` that:
- Tokenises `text` by whitespace (`split()`)
- Yields overlapping chunks of `chunk_size` tokens
- Stride = `chunk_size - overlap` (consecutive chunks share `overlap` tokens)
- Handles `chunk_size >= len(tokens)` gracefully — yields the whole text as one chunk

**Example:**
```
text = "a b c d e f g", chunk_size=3, overlap=1
tokens = ["a", "b", "c", "d", "e", "f", "g"]
stride  = 3 - 1 = 2

i=0: tokens[0:3] = ["a", "b", "c"] → "a b c"
i=2: tokens[2:5] = ["c", "d", "e"] → "c d e"
i=4: tokens[4:7] = ["e", "f", "g"] → "e f g"
```

<details>
<summary>Hint (expand only after a genuine attempt)</summary>

Index-based slicing works fine here — tokens are already in a list. `stride = chunk_size - overlap`. Loop: `for i in range(0, len(tokens), stride)`. Slice: `tokens[i : i + chunk_size]`. Join the slice with `" ".join(slice)`. Handle the edge case first: `if chunk_size >= len(tokens): yield text; return`.

</details>

In [ ]:
import inspect

def chunk_text(text: str, chunk_size: int, overlap: int):
    """
    Generator: yields overlapping token chunks from text.
    Tokenises by whitespace. Stride = chunk_size - overlap.
    If chunk_size >= len(tokens), yields the entire text as one chunk.
    """
    # TODO: implement this generator
    ...


# ── Verify block ──────────────────────────────────────────────────────────────
try:
    assert inspect.isgeneratorfunction(chunk_text), "chunk_text must be a generator function (use yield)"
    print("Requirement 1: PASSED — chunk_text is a generator function")

    result = list(chunk_text("a b c d e f g", chunk_size=3, overlap=1))
    assert result == ["a b c", "c d e", "e f g"], f"Expected ['a b c', 'c d e', 'e f g'], got {result}"
    print("Requirement 2: PASSED — correct overlapping chunks (stride=2)")

    short_result = list(chunk_text("governing law", chunk_size=5, overlap=1))
    assert short_result == ["governing law"], f"Short text: expected ['governing law'], got {short_result}"
    print("Requirement 3: PASSED — chunk_size > text length handled gracefully")

    no_overlap = list(chunk_text("a b c d e f", chunk_size=2, overlap=0))
    assert no_overlap == ["a b", "c d", "e f"], f"No-overlap: expected ['a b', 'c d', 'e f'], got {no_overlap}"
    print("Requirement 4: PASSED — zero overlap (non-overlapping chunks)")

    print("Score: 4/4")
except NotImplementedError:
    print("Not implemented — write your generator in chunk_text()")
except AssertionError as e:
    print(f"FAILED: {e}")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

---
## Consolidation pause 3 — after Part 5

Step away and answer these without looking at the notebook.

**Q1.** Why do you use `islice(read_jsonl(path), 100)` instead of `list(read_jsonl(path))[:100]`?

**Q2.** At which stage of a generator pipeline is it acceptable to call `list()` on a generator?

**Q3.** `deque(maxlen=3)` currently holds `["NDA", "MSA", "SOW"]`. You `append("RA-001")`. What does the deque hold?

**Q4.** Recall the Part 5 sticky phrase: "Compose generators. Each stage ___. `itertools` is ___."

---

<details>
<summary>Expected answers</summary>

**A1.** `islice` is lazy — it pulls items one at a time until it has taken n, then stops. `list(read_jsonl(path))` loads the entire 50 GB file into memory before slicing. The slice is never worth the cost.

**A2.** Only at the sink — when you need all results simultaneously (test assertions, passing a list to an API). Never mid-pipeline. Materialise as late as possible.

**A3.** `deque(["MSA", "SOW", "RA-001"])` — `"NDA"` is automatically dropped because `maxlen=3` is enforced on every `append`.

**A4.** "Compose generators. Each stage **yields one item**. `itertools` is **the glue**."

</details>

---
## Final exercise — Marcus's 4 PM code review

**Slack — Marcus Okafor → you, 3:55 PM**

> "Ready. Send me both PRs — `streaming_reader.py` and `chunker.py`. I'll run the review script live. Score 6/6 and both are approved. Score below that and I'll tell you exactly what to fix."

---

### Worked example — the structural template for `read_jsonl`

Before you write your final implementation, here is the complete correct structure. This is what Marcus expects — study it before writing your own.

<details>
<summary>Reference implementation (expand to study before writing yours)</summary>

```python
import json
import sys

def read_jsonl(path: str):
    """
    Generator: yields one parsed dict per line from a JSONL file.
    Skips empty lines silently.
    Warns on JSONDecodeError and continues — does not raise.
    """
    with open(path) as f:
        for line in f:                         # lazy file iteration
            line = line.strip()
            if not line:
                continue                       # skip blanks
            try:
                yield json.loads(line)         # pause here, return dict
            except json.JSONDecodeError as e:
                print(f"Warning: {path}: {e}", file=sys.stderr)
                # continue to next line — do not propagate the error
```

Key points:
1. `with open(path) as f:` stays open across all `yield` calls — the file handle is preserved in the generator's local state.
2. `yield` pauses the function. The next `next()` call resumes from the line immediately after `yield`.
3. `print(..., file=sys.stderr)` goes to stderr — does not pollute the record stream.
4. No `list()`, no `readlines()`, no `read()` — the file is never fully loaded.

</details>

---

Now implement both functions below, then run Marcus's review script.

In [ ]:
import json
import sys
import io
import inspect


# ── PR 1: ingestion/streaming_reader.py ───────────────────────────────────────
def read_jsonl_pr(path: str):
    """
    Generator: yields one parsed dict per line from a JSONL file.
    Skips empty lines silently.
    Warns on JSONDecodeError and continues — does not raise.
    """
    # TODO: implement using the worked example as your template
    ...


# ── PR 2: ingestion/chunker.py ────────────────────────────────────────────────
def chunk_text_pr(text: str, chunk_size: int, overlap: int):
    """
    Generator: yields overlapping text chunks.
    Tokenises by whitespace. Stride = chunk_size - overlap.
    Yields entire text if chunk_size >= len(tokens).
    """
    # TODO: implement (you can reuse your chunk_text implementation above)
    ...

In [ ]:
# ── Marcus's review script — runs automatically ───────────────────────────────
score = 0
total = 6

print("Marcus's 4 PM code review")
print("=" * 44)

# Test 1: read_jsonl_pr is a generator function
try:
    assert inspect.isgeneratorfunction(read_jsonl_pr), "read_jsonl_pr must use yield"
    print("Test 1: PASSED — streaming_reader uses yield (lazy generator)")
    score += 1
except AssertionError as e:
    print(f"Test 1: FAILED — {e}")

# Test 2: Parses valid JSONL records
test_data = (
    '{"contract_id": "NDA-001", "firm": "Brennan & Carson", "status": "pending"}\n'
    '{"contract_id": "MSA-002", "firm": "Google LLC", "status": "processed"}\n'
)
try:
    # Apply read_jsonl logic to StringIO (no file required)
    def _stream_parse(text):
        for line in io.StringIO(text):
            line = line.strip()
            if not line:
                continue
            try:
                yield json.loads(line)
            except json.JSONDecodeError:
                pass

    records = list(_stream_parse(test_data))
    assert len(records) == 2 and records[0]["contract_id"] == "NDA-001"
    print("Test 2: PASSED — parses 2 valid contract records")
    score += 1
except AssertionError as e:
    print(f"Test 2: FAILED — {e}")

# Test 3: Skips empty lines and malformed JSON
mixed_data = (
    '{"contract_id": "SOW-003"}\n'
    '\n'
    'MALFORMED\n'
    '{"contract_id": "NDA-004"}\n'
)
try:
    records2 = list(_stream_parse(mixed_data))
    assert len(records2) == 2, f"Expected 2 valid records, got {len(records2)}"
    print("Test 3: PASSED — empty lines and malformed JSON handled correctly")
    score += 1
except AssertionError as e:
    print(f"Test 3: FAILED — {e}")

# Test 4: chunk_text_pr is a generator function
try:
    assert inspect.isgeneratorfunction(chunk_text_pr), "chunk_text_pr must use yield"
    print("Test 4: PASSED — chunker uses yield (lazy generator)")
    score += 1
except AssertionError as e:
    print(f"Test 4: FAILED — {e}")

# Test 5: Correct overlapping chunks
try:
    result = list(chunk_text_pr("a b c d e f g", chunk_size=3, overlap=1))
    assert result == ["a b c", "c d e", "e f g"], f"Got: {result}"
    print("Test 5: PASSED — overlapping chunks with stride=2 correct")
    score += 1
except (AssertionError, TypeError) as e:
    print(f"Test 5: FAILED — {e}")

# Test 6: Edge case — text shorter than chunk_size
try:
    short_result = list(chunk_text_pr("governing law", chunk_size=5, overlap=1))
    assert short_result == ["governing law"], f"Got: {short_result}"
    print("Test 6: PASSED — short text yields whole text as one chunk")
    score += 1
except (AssertionError, TypeError) as e:
    print(f"Test 6: FAILED — {e}")

print()
print(f"Score: {score}/{total}")

if score == total:
    print()
    print("Marcus: Both PRs approved. The streaming reader will handle the Lambda memory limit.")
    print("The chunker is what we pass into the embedding pipeline next week.")
    print("This is production-quality. Priya will see it at Friday's checkpoint.")
else:
    print()
    print(f"Marcus: {total - score} test(s) failing. Fix before end of day.")

---
## Module summary — screenshot this

| Concept | Sticky phrase | Quick test |
|---|---|---|
| Data structure selection | "Right structure, right operation. `in` a list is O(n). `in` a set is O(1). At scale, this is the only decision that matters." | When does `deque` beat `list`? |
| Comprehensions | "Comprehensions for clarity. Generator expressions for memory. Nested 3-deep? Use a loop." | How do you get the first match without evaluating all items? |
| Iterator protocol | "`for` calls `iter()` then keeps calling `next()` until `StopIteration`. Any object that speaks this protocol is iterable." | What happens on `next()` after exhaustion? |
| Generators | "A generator holds one object at a time. `yield` pauses. `next()` resumes. Memory stays flat." | After `gen = read_jsonl(path)`, has the file opened? |
| Generator composition | "Compose generators. Each stage yields one item. `itertools` is the glue." | Why is `list()` mid-pipeline a mistake? |

---

**The 50 GB answer — say it cold:**
> A generator yields one object at a time. After each record is processed, no label points to it — the garbage collector reclaims it before the next record is created. Memory stays at ~1 record regardless of file size. A list creates all 40M dicts simultaneously: 40M × 250 bytes = 10 GB. The Lambda has 512 MB. OOM.

---
## What comes next — Module 01_01_03: Functions, Closures & Packages

The `read_jsonl` generator you built today is a closure. The generator object holds a reference to the open file handle and the current line position — a function body combined with its enclosing environment. That is the definition of a closure.

In Module 01_01_03 you build closure-based patterns explicitly: `make_token_bucket` (a rate limiter that counts tokens across calls), `make_retry` (a retry wrapper with exponential backoff), and a token budget enforcer. Every one of them uses the same mechanism — a function that captures a reference to an enclosing variable and holds state between calls.

The `nonlocal` keyword, which you use in Module 1.3, is exactly what makes that enclosing variable mutable. Without it, `count += 1` inside a closure creates a new local variable and raises `UnboundLocalError`. Module 1.3 is where you name this mechanism explicitly and build production-grade utilities on top of it.

---

**Week 1 checkpoint (01_01_CK):** After Module 01_01_03, the checkpoint asks you to build the `jsonl-filter` CLI — a fully packaged tool that uses `read_jsonl` from this module, the closure utilities from Module 1.3, and ships as a `uv`-installable CLI. That deliverable is what Priya reviews on Friday.